# 05 — Non-Overlapping Significance Check: Is the 120-Day IC Real?

The horizon sweep showed a striking result: rank IC rising from ~0.014 (5-day) to
**~0.087 (120-day, Ridge)**, positive in every fold. That is a big jump — big enough
to justify paid fundamentals data *if it is real*. This notebook stress-tests whether
it is.

## The problem this checks

At a 120-day horizon, two observations one trading day apart share **119 of their 120
outcome days**. So the ~1.9M rows are almost entirely redundant: the number of
*independent* 120-day returns is roughly

    16 years / 120 trading days ≈ 33 non-overlapping periods (total, across all folds).

An IC computed on 1.9M overlapping rows *looks* precise, but its real precision comes
from ~33 independent periods, not 1.9M. Overlapping labels are the classic way a
mediocre long-horizon signal masquerades as a strong one: nominal sample huge,
effective sample tiny.

## What this notebook does

1. **Non-overlapping resample.** Keep one observation per horizon-length block per
   ticker, so every retained row is (approximately) independent. Recompute the IC on
   this honest, much smaller sample.
2. **Block-level significance.** Treat each rebalance date's cross-sectional IC as one
   independent draw; test whether the *mean* of those per-period ICs is distinguishable
   from zero (t-stat on the period ICs, not on rows).
3. **Compare** the overlapping IC (inflated) against the non-overlapping IC (honest).

**If the non-overlapping IC stays meaningful (say >= 0.03-0.04) with a decent t-stat**
-> the signal is real and paid fundamentals are justified. **If it collapses toward
zero** -> the 0.087 was largely an overlapping-label artifact, and the return ceiling
is lower than the sweep suggested.

In [ ]:
from pathlib import Path
import sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, ttest_1samp
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True; plt.rcParams["grid.alpha"] = 0.3
warnings.filterwarnings("ignore")

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
for parent in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (parent / 'src').is_dir():
        PROJECT_ROOT = parent; break
else:
    raise RuntimeError('Run from inside the StockForecastRisk repository.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.forecast_engine.features.schema import FEATURE_NAMES, NON_FEATURE_COLUMNS
from src.forecast_engine.data.loader import load_processed_features

data = load_processed_features()
data["date"] = pd.to_datetime(data["date"])
data = data.sort_values(["symbol", "date"]).reset_index(drop=True)

NON_STATIONARY_LEVELS = ["sma_10","sma_20","sma_50","sma_200","ema_12","ema_26","vwap_20"]
SI = ["short_interest","short_interest_change","days_to_cover","days_to_cover_change"]
EXCLUDE = set(NON_FEATURE_COLUMNS) | set(NON_STATIONARY_LEVELS) | set(SI) | {"short_history","history_rows"}
feature_cols = [c for c in FEATURE_NAMES if c not in EXCLUDE and c in data.columns and not data[c].isna().all()]

HORIZON = 120  # the horizon under scrutiny (also run 60 below)
print(f"features: {len(feature_cols)}, horizon under test: {HORIZON}d")

In [ ]:
def forward_log_return(group, h):
    c = group["adj_close"].astype(float)
    return np.log(c.shift(-h) / c)

for h in (60, 120):
    data[f"fwd_ret_{h}"] = data.groupby("symbol", group_keys=False).apply(
        lambda g: forward_log_return(g, h))
print("targets built for 60d, 120d")

## Walk-forward predictions (Ridge), then two ways of scoring the IC

We generate out-of-sample Ridge predictions with the same purged walk-forward as the
sweep. Then we score the IC two ways on the *same* predictions:

- **Overlapping (all rows):** every trading day — the inflated version from the sweep.
- **Non-overlapping (every Hth date):** keep only rebalance dates spaced `HORIZON`
  trading days apart, so outcome windows do not overlap. This is the honest version.

In [ ]:
N_SPLITS = 5
def walk_forward_splits(frame, purge_days):
    dates = np.sort(frame["date"].unique())
    edges = np.array_split(dates, N_SPLITS + 1)
    for f in range(N_SPLITS):
        cutoff = edges[f][-1] - pd.Timedelta(days=purge_days * 2)
        tr = frame.index[frame["date"] <= cutoff]
        te = frame.index[frame["date"].isin(edges[f + 1])]
        if len(tr) and len(te):
            yield tr, te

def oos_predictions(h):
    target = f"fwd_ret_{h}"
    md_h = data.dropna(subset=feature_cols + [target]).reset_index(drop=True)
    X = md_h[feature_cols].to_numpy(np.float32)
    y = md_h[target].to_numpy(np.float32)
    preds = np.full(len(md_h), np.nan, dtype=np.float32)
    for tr, te in walk_forward_splits(md_h, purge_days=h):
        sc = StandardScaler().fit(X[tr])
        preds[te] = Ridge(alpha=1.0).fit(sc.transform(X[tr]), y[tr]).predict(sc.transform(X[te]))
    out = md_h[["date", "symbol", target]].copy()
    out["pred"] = preds
    return out.dropna(subset=["pred"])

def per_date_ic(frame, target):
    """Cross-sectional IC on each date; returns a Series indexed by date."""
    recs = {}
    for d, g in frame.groupby("date"):
        if g[target].nunique() > 2 and g["pred"].nunique() > 2:
            c = spearmanr(g["pred"], g[target]).correlation
            if np.isfinite(c):
                recs[d] = c
    return pd.Series(recs).sort_index()

print("helpers ready")

In [ ]:
def analyse(h):
    target = f"fwd_ret_{h}"
    oos = oos_predictions(h)
    ic_by_date = per_date_ic(oos, target)

    # Overlapping: mean over ALL dates (what the sweep reported).
    overlapping_ic = ic_by_date.mean()

    # Non-overlapping: keep dates spaced >= h trading days apart, so outcome
    # windows do not overlap. Greedily walk the sorted dates.
    kept = []
    last = None
    for d in ic_by_date.index:
        if last is None or (d - last).days >= h * (7/5):  # h trading days in calendar days
            kept.append(d); last = d
    nonoverlap_ic = ic_by_date.loc[kept]

    # Significance: treat each retained period's IC as one independent draw.
    n = len(nonoverlap_ic)
    mean_ic = nonoverlap_ic.mean()
    if n > 1:
        t, p = ttest_1samp(nonoverlap_ic, 0.0)
    else:
        t, p = np.nan, np.nan

    return {
        "horizon": h,
        "overlapping_ic": overlapping_ic,
        "n_overlapping_dates": len(ic_by_date),
        "nonoverlap_ic": mean_ic,
        "n_independent_periods": n,
        "t_stat": t,
        "p_value": p,
        "ic_series_full": ic_by_date,
        "ic_series_indep": nonoverlap_ic,
    }

res60 = analyse(60)
res120 = analyse(120)

summary = pd.DataFrame([
    {k: v for k, v in r.items() if not k.startswith("ic_series")}
    for r in (res60, res120)
]).set_index("horizon")
print("Overlapping vs non-overlapping IC, with significance on independent periods:")
display(summary.round(4))

In [ ]:
# Verdict print
for r in (res60, res120):
    h = r["horizon"]
    print(f"\n=== {h}-day horizon ===")
    print(f"  overlapping IC (sweep number) : {r['overlapping_ic']:+.4f}  "
          f"(over {r['n_overlapping_dates']} dates -- inflated, redundant)")
    print(f"  non-overlapping IC (honest)   : {r['nonoverlap_ic']:+.4f}  "
          f"(over {r['n_independent_periods']} INDEPENDENT periods)")
    print(f"  t-stat / p-value              : {r['t_stat']:+.2f} / {r['p_value']:.3f}")
    if r["n_independent_periods"] < 2 or not np.isfinite(r["t_stat"]):
        print("  -> too few independent periods to test -- treat as unproven.")
    elif r["p_value"] < 0.05 and r["nonoverlap_ic"] > 0.02:
        print("  -> SURVIVES: signal is real on independent data. Fundamentals bet justified.")
    elif r["nonoverlap_ic"] > 0.02:
        print("  -> IC holds but not significant (few periods). Promising, not proven.")
    else:
        print("  -> COLLAPSES: the sweep IC was largely an overlapping-label artifact.")

In [ ]:
# Visual: the per-period IC distribution, overlapping vs independent.
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, r in zip(axes, (res60, res120)):
    h = r["horizon"]
    ax.hist(r["ic_series_full"].values, bins=40, alpha=0.4, color="tab:blue",
            label=f"all dates (n={r['n_overlapping_dates']})", density=True)
    ax.hist(r["ic_series_indep"].values, bins=15, alpha=0.6, color="tab:orange",
            label=f"independent (n={r['n_independent_periods']})", density=True)
    ax.axvline(0, color="red", ls="--")
    ax.axvline(r["nonoverlap_ic"], color="tab:orange", lw=2,
               label=f"indep mean {r['nonoverlap_ic']:+.3f}")
    ax.set_title(f"{h}-day per-period IC: overlapping vs independent")
    ax.set_xlabel("per-date rank IC"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Independent-period IC over time -- is it consistently positive, or regime-lumpy?
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, r in zip(axes, (res60, res120)):
    s = r["ic_series_indep"]
    colors = ["tab:green" if v > 0 else "tab:red" for v in s.values]
    ax.bar(range(len(s)), s.values, color=colors, alpha=0.8)
    ax.axhline(0, color="black", lw=0.8)
    ax.axhline(s.mean(), color="blue", ls="--", label=f"mean {s.mean():+.3f}")
    ax.set_title(f"{r['horizon']}-day IC per independent period "
                 f"({(s>0).sum()}/{len(s)} positive)")
    ax.set_xlabel("independent period"); ax.set_ylabel("rank IC"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## Conclusion — is the long-horizon return signal real?

Fill from the numbers above.

- **120-day: overlapping IC ____ vs non-overlapping IC ____** — how much did it shrink?
- **Independent periods: ____** — how small is the honest sample?
- **t-stat / p-value: ____** — distinguishable from zero, or not?
- **Fraction of independent periods positive: ____** — consistent sign, or coin-flip?

### The decision on paid fundamentals

- **Non-overlapping IC stays >= ~0.03-0.04 with p < 0.05 and mostly-positive periods**
  -> the long-horizon return signal is REAL, not an overlap artifact. Paid fundamentals
  (Sharadar) are justified: they are 120-day-appropriate signals that would add to a
  horizon that genuinely works. Reframe the return model to 120-day Ridge and proceed.

- **Non-overlapping IC collapses toward zero, or the t-stat is weak, or the periods are
  a coin-flip** -> the 0.087 was mostly overlapping-label inflation. The honest return
  ceiling on price data is low. Do NOT spend on fundamentals expecting a return lift;
  ship the volatility engine (IC 0.48, validated) as the deliverable and keep returns as
  a thin tilt at best.

Either way, this is the discipline that matters: the more impressive a result looks, the
harder it must be tested before money or belief is committed. This is that test.